In [9]:
import json
import pandas as pd
from pathlib import Path

In [ ]:
INPUT_JSONL = Path("merged_culture_dataset.jsonl")
OUTPUT_CSV = Path("merged_culture_dataset_as_markets_format.csv")

print("INPUT_JSONL =", INPUT_JSONL.resolve())
print("OUTPUT_CSV  =", OUTPUT_CSV.resolve())

INPUT_JSONL = C:\Users\Zver\Desktop\нейронки\проект\merged_culture_dataset.jsonl
OUTPUT_CSV  = C:\Users\Zver\Desktop\нейронки\проект\merged_culture_dataset_as_markets_format.csv


In [ ]:
rows = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

print("Строк загружено:", len(df))
print("Колонки входного файла:")
print(df.columns.tolist())
df.head(3)

Строк загружено: 1996
Колонки входного файла:
['id', 'domain', 'subcategory', 'question_type', 'question', 'prediction_date', 'resolution_date', 'answer', 'resolution_reasoning', 'prompt']


,id,domain,subcategory,question_type,question,prediction_date,resolution_date,answer,resolution_reasoning,prompt
0,92a7a134-5fa7-4b78-890d-779cef496cc1,culture,awards,awards,Will the 79th Golden Globe Awards (2022) be br...,2022-01-07T00:00:00,2022-01-09T00:00:00,NO,The 79th Golden Globe Awards took place on Jan...,[]
1,9d852b9e-707d-40a9-a6d7-9de0c07c03ac,culture,awards,other,Will the cast of Yellowstone win the Screen Ac...,2022-01-12T00:00:00,2022-02-27T00:00:00,NO,The 28th Annual Screen Actors Guild (SAG) Awar...,[]
2,eaff58ba-7beb-4c64-a4ec-a83871d08e11,culture,awards,awards,Will Jennifer Hudson receive a 2024 Daytime Em...,2022-01-15T00:00:00,2024-04-19T00:00:00,NO,The official nominations for the 51st Annual D...,[]


In [ ]:
def normalize_answer(ans):
    if pd.isna(ans):
        return None
    s = str(ans).strip().lower()
    if s in {"yes", "true", "1"}:
        return "YES"
    if s in {"no", "false", "0"}:
        return "NO"
    return None

df["answer_norm"] = df["answer"].apply(normalize_answer)
df["answer_norm"].value_counts(dropna=False)

answer_norm
YES    1060
NO      936
Name: count, dtype: int64

In [ ]:
#форматируем под полимаркет
out = pd.DataFrame({
    "question": df["question"],
    "startDate": df["prediction_date"],
    "endDate": df["resolution_date"],
    "umaResolutionStatus": df["answer_norm"].apply(
        lambda x: "resolved" if x in {"YES", "NO"} else "unresolved"
    ),
    "winner_outcome": df["answer_norm"].apply(
        lambda x: "Yes" if x == "YES" else ("No" if x == "NO" else None)
    ),
    "final_outcome": df["answer_norm"].apply(
        lambda x: 1 if x == "YES" else (0 if x == "NO" else None)
    ),
    "resolution_reasoning": df["resolution_reasoning"]
})

print("Колонки итогового файла:")
print(out.columns.tolist())
out.head(5)

Колонки итогового файла:
['question', 'startDate', 'endDate', 'umaResolutionStatus', 'winner_outcome', 'final_outcome', 'resolution_reasoning']


,question,startDate,endDate,umaResolutionStatus,winner_outcome,final_outcome,resolution_reasoning
0,Will the 79th Golden Globe Awards (2022) be br...,2022-01-07T00:00:00,2022-01-09T00:00:00,resolved,No,0,The 79th Golden Globe Awards took place on Jan...
1,Will the cast of Yellowstone win the Screen Ac...,2022-01-12T00:00:00,2022-02-27T00:00:00,resolved,No,0,The 28th Annual Screen Actors Guild (SAG) Awar...
2,Will Jennifer Hudson receive a 2024 Daytime Em...,2022-01-15T00:00:00,2024-04-19T00:00:00,resolved,No,0,The official nominations for the 51st Annual D...
3,Will Kevin Costner win the Golden Globe Award ...,2022-01-18T00:00:00,2023-01-10T00:00:00,resolved,Yes,1,Kevin Costner won the Golden Globe Award for B...
4,Will Amazon MGM Studios or a major streaming s...,2022-01-19T00:00:00,2025-12-31T00:00:00,resolved,No,0,The close date for this question was 2025-12-3...


In [ ]:
print("Итоговых строк перед сохранением:", len(out))
out["umaResolutionStatus"].value_counts(dropna=False)

Итоговых строк перед сохранением: 1996


umaResolutionStatus
resolved    1996
Name: count, dtype: int64

In [ ]:
out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Готово.")
print("CSV сохранён сюда:")
print(OUTPUT_CSV.resolve())

Готово.
CSV сохранён сюда:
C:\Users\Zver\Desktop\нейронки\проект\merged_culture_dataset_as_markets_format.csv


In [ ]:
check = pd.read_csv(OUTPUT_CSV)
print("Размер:", check.shape)
check.head(10)

Размер: (1996, 7)


,question,startDate,endDate,umaResolutionStatus,winner_outcome,final_outcome,resolution_reasoning
0,Will the 79th Golden Globe Awards (2022) be br...,2022-01-07T00:00:00,2022-01-09T00:00:00,resolved,No,0,The 79th Golden Globe Awards took place on Jan...
1,Will the cast of Yellowstone win the Screen Ac...,2022-01-12T00:00:00,2022-02-27T00:00:00,resolved,No,0,The 28th Annual Screen Actors Guild (SAG) Awar...
2,Will Jennifer Hudson receive a 2024 Daytime Em...,2022-01-15T00:00:00,2024-04-19T00:00:00,resolved,No,0,The official nominations for the 51st Annual D...
3,Will Kevin Costner win the Golden Globe Award ...,2022-01-18T00:00:00,2023-01-10T00:00:00,resolved,Yes,1,Kevin Costner won the Golden Globe Award for B...
4,Will Amazon MGM Studios or a major streaming s...,2022-01-19T00:00:00,2025-12-31T00:00:00,resolved,No,0,The close date for this question was 2025-12-3...
5,Will the first season of the Paramount+ series...,2022-01-20T00:00:00,2023-06-01T00:00:00,resolved,No,0,"The close date: 2023-06-01, the question date:..."
6,Will Ben Platt perform a concert at the Peters...,2022-02-02T00:00:00,2023-01-01T00:00:00,resolved,No,0,Ben Platt's 'Reverie' tour was originally anno...
7,Will 'Spider-Man: No Way Home' win the 'Fan Fa...,2022-02-15T00:00:00,2022-03-27T00:00:00,resolved,No,0,"At the 94th Academy Awards held on March 27, 2..."
8,Will the documentary 'Lucy and Desi' (directed...,2022-02-15T00:00:00,2022-07-12T00:00:00,resolved,Yes,1,The 74th Primetime Emmy Awards nominations wer...
9,Will the first episode of Yellowstone Season 5...,2022-02-24T00:00:00,2024-11-10T00:00:00,resolved,Yes,1,The second half of Yellowstone Season 5 (Seaso...
